# HackatEOn

## Agenda

**Classification**

1. Meet the data
    1. Biromaru, Indonesia
    2. Grande-Terre, Guadeloupe
    3. Segmentation Masks
2. Classification task
    1. Random Forest Classifier
    2. Support Vector Machine
    3. Artificial Neural Network
    4. Data Preprocessing
    5. Spectral Information

**Segmentation**

3. Segmentation task
    1. Random Forest
    2. Fully Convolutional Network (FCN)
    3. U-Net
    4. Pre-trained Models
    5. Transfer Learning: Fine Tuning
    6. Transfer Learning: Feature Extraction
    7. Class Imbalance

In [ ]:
import h5py
import numpy as np
import math
import pandas as pd

import gc

import matplotlib.pyplot as plt
from matplotlib import cm, colormaps
from matplotlib.ticker import PercentFormatter

from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchinfo import summary

from functools import reduce

from IPython.display import IFrame

np.random.seed(3108)

## 1. Meet the data

In [ ]:
# this should match the name of the data sets
filename = '../data/dynamic_earth_data.h5'

dataset = h5py.File(filename, 'r')

# show the content names
print(list(dataset.keys()))

### 1.1. Biromaru, Indonesia

In [ ]:
IFrame("https://www.google.com/maps/embed?pb=!1m18!1m12!1m3!1d17835.880996110664!2d119.89901399056846!3d-0.9615886027399949!2m3!1f0!2f0!3f0!3m2!1i1024!2i768!4f13.1!3m3!1m2!1s0x2d8bf75fadf31657%3A0xca14d076a8f526b9!2sSigi%20Biromaru%2C%20Sigi%20Regency%2C%20Central%20Sulawesi%2C%20Indonesia!5e0!3m2!1sen!2sde!4v1791468642275!5m2!1sen!2sde", 
       width="100%", height=400)

In [ ]:
# load Sentinel-2 data
sen2_train = np.array(dataset['biromaru_s2'])

print("Sentinel-2 shape: ", sen2_train.shape)
# 24 samples, 1024x1024 pixels, 12 bands


| File Order | Band Identifier | Band Name | GSD (Resolution) | Central Wavelength (nm) |
| :---: | :--- | :--- | :---: | :---: |
| **1** | B01 | Coastal Aerosol | 60 meters | 443 |
| **2** | B02 | Blue | 10 meters | 490 |
| **3** | B03 | Green | 10 meters | 560 |
| **4** | B04 | Red | 10 meters | 665 |
| **5** | B05 | Red Edge 1 | 20 meters | 705 |
| **6** | B06 | Red Edge 2 | 20 meters | 740 |
| **7** | B07 | Red Edge 3 | 20 meters | 783 |
| **8** | B08 | NIR (Near Infrared) | 10 meters | 842 |
| **9** | B8A | Narrow NIR | 20 meters | 865 |
| **10** | B09 | Water Vapor | 60 meters | 945 |
| **11** | B11 | SWIR 1 (Short-Wave IR) | 20 meters | 1610 |
| **12** | B12 | SWIR 2 (Short-Wave IR) | 20 meters | 2190 |


**NOTE**: The dataset upscaled all bands to 3 m GSD.

In [ ]:
plt.hist(sen2_train[:,:,:,0].flatten(), color="blue", label="B02", alpha=0.7, bins=100)
plt.hist(sen2_train[:,:,:,1].flatten(), color="green", label="B03", alpha=0.7, bins=100)
plt.hist(sen2_train[:,:,:,2].flatten(), color="red", label="B04", alpha=0.7, bins=100)

cmap = colormaps['plasma']
plasma_colors = cmap(np.linspace(0, 1, 9))

band_labels = ["B08", "B05", "B6", "B07", "B8A", "B11", "B12", "B01", "B09"]

for band in range(3,12):
    plt.hist(sen2_train[:,:,:,band].flatten(), color=plasma_colors[band-3,], label=band_labels[band-3], alpha=0.2, bins=100)
plt.yscale('symlog')
plt.legend()
plt.show()

Note: The pixel values are *digital numbers* (DN), and can be converted into surfance reflectance.

$$Reflectance=\frac{DN}{10000}$$

Higher values are expected in infrared bands. Reflectance can be higher than 1. More info at https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-2-msi/


In [ ]:
def true_color(X):
    # B2 = blue, B3 = green, and B4 = red
    blue = ((X[:,:,0] - np.min(X[:,:,0])) / (np.max(X[:,:,0]) - np.min(X[:,:,0])))
    green = ((X[:,:,1] - np.min(X[:,:,1])) / (np.max(X[:,:,1]) - np.min(X[:,:,1])))
    red = ((X[:,:,2] - np.min(X[:,:,2])) / (np.max(X[:,:,2]) - np.min(X[:,:,2])))
    return np.dstack((red, green, blue))  

index = 2

fig=plt.figure(figsize=(12, 12))


plt.imshow(true_color(sen2_train[index]))
plt.title('Sentinel-2')
plt.show()

In [ ]:
# load Planet Fusion data
planet_train = np.array(dataset['biromaru_planet'])

print("Planet Fusion shape: ", planet_train.shape)
# 24 samples, 1024x1024 pixels, 4 bands


| File Order | Band Identifier | Band Name | GSD (Resolution) | Central Wavelength (nm) | Harmonized To |
| :---: | :--- | :--- | :---: | :---: | :--- |
| **1** | Band 1 | Blue | 3 meters | 490 | Sentinel-2 (B02) |
| **2** | Band 2 | Green | 3 meters | 560 | Sentinel-2 (B03) |
| **3** | Band 3 | Red | 3 meters | 665 | Sentinel-2 (B04) |
| **4** | Band 4 | NIR (Near Infrared) | 3 meters | 865 | Sentinel-2 (B8A) |

In [ ]:
plt.hist(planet_train[:,:,:,0].flatten(), color="blue", label="blue", alpha=0.5, bins=100)
plt.hist(planet_train[:,:,:,1].flatten(), color="green", label="green", alpha=0.5, bins=100)
plt.hist(planet_train[:,:,:,2].flatten(), color="red", label="red", alpha=0.5, bins=100)
plt.hist(planet_train[:,:,:,3].flatten(), color="orange", label="nir", alpha=0.2, bins=100)

#plt.yscale('symlog')
plt.legend()
plt.show()

The data values (converted to DN) are *analysis-ready* surface reflectance, harmonized to Sentinel-1 standards.

In [ ]:
fig=plt.figure(figsize=(12, 12))
plt.imshow(true_color(planet_train[index]))
plt.title('Planet Scope')
plt.show()

In [ ]:
# load Sentinel-1 data
sen1_train = np.array(dataset['biromaru_s1'])

print("Sentinel-1 shape: " + str(sen1_train.shape))


The data set (Sentinel-1 SLC) contains the following channels:
1) the real part of the unfiltered VH channel
2) the imaginary part of the unfiltered VH channel
3) the real part of the unfiltered VV channel
4) the imaginary part of the unfiltered VV channel
5) the intensity of the refined Lee filtered VH channel
6) the intensity of the refined Lee filtered VV channel
7) the real part of the refined Lee filtered covariance matrix off-diagonal element
8) the imaginary part of the refined Lee filtered covariance matrix off-diagonal element

In [ ]:
def false_color(X):
    VH = np.sqrt(X[:,:,0]**2 + X[:,:,1]**2) 
    VV = np.sqrt(X[:,:,2]**2 + X[:,:,3]**2)
    
    R = 20 * np.log10(VV + 1e-5)
    G = 20 * np.log10(VH + 1e-5)
    B = R - G

    R = (R - R.min()) / (R.max() - R.min())
    G = (G - G.min()) / (G.max() - G.min())
    B = (B - B.min()) / (B.max() - B.min())
        
    tc = np.dstack((R, G, B))
    
    return tc

plt.imshow(false_color(sen1_train[index,:,:,:]))
plt.title('Sentinel-1')

plt.show()

### 1.2. Grande-Terre, Guadeloupe

In [ ]:
IFrame("https://www.google.com/maps/embed?pb=!1m18!1m12!1m3!1d51691.2949337503!2d-61.48382083756777!3d16.483862619942023!2m3!1f0!2f0!3f0!3m2!1i1024!2i768!4f13.1!3m3!1m2!1s0x8c1335dee5121b95%3A0x6baaa206604e9893!2sGrande-Terre!5e1!3m2!1sen!2sde!4v1791463471452!5m2!1sen!2sde", 
       width="100%", height=400)

In [ ]:
sen2_test = np.array(dataset['guadeloupe_s2'])
planet_test = np.array(dataset['guadeloupe_planet'])

fig=plt.figure(figsize=(12, 6))
fig.add_subplot(1, 2, 1)
plt.imshow(true_color(sen2_test[index]))
plt.title('Sentinel-2')
fig.add_subplot(1, 2, 2)
plt.imshow(true_color(planet_test[index]))
plt.title('Planet Scope')
plt.show()

#### Do it yourself

Check the data distribution (e.g., plot the histograms). Are both *AOI*s "similar"?

### 1.3. Segmentation Masks

In [ ]:
# load Planet Fusion data
labels_train = np.array(dataset['biromaru_labels'])
labels_test = np.array(dataset['guadeloupe_labels'])

print("Labels shape: ", labels_train.shape)
# 24 samples, 1024x1024 pixels, 7 dimensions


In [ ]:
print("One-hot encoded pixel", labels_train[0,0,0,:])

In [ ]:
color_map = cm.get_cmap("Accent", lut=labels_train.shape[-1])
labels=['impervious surface',
       'agriculture',
       'forest & other vegetation',
       'wetlands',
       'soil',
       'water',
       'snow & ice']

def labels2rgb(
    X,
    colors=color_map.colors,
    categorical=False):
    Z = np.zeros((X.shape[0], X.shape[1], 3))
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            if categorical:
                Z[i,j] = colors[int(X[i,j]), :3]
            else:
                Z[i,j] = colors[np.argmax(X[i,j]), :3]
    return Z

fig=plt.figure(figsize=(12, 12))
plt.imshow(labels2rgb(labels_train[index]), cmap=color_map)
plt.title('Labels')
cbar = plt.colorbar(ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)
plt.show()

In [ ]:
fig=plt.figure(figsize=(12, 12))
plt.imshow(labels2rgb(labels_test[index]), cmap=color_map)
plt.title('Labels')
cbar = plt.colorbar(ticks=np.linspace(1/14, 13/14, 7), shrink=0.7)
cbar.set_ticklabels(labels)
plt.show()

In [ ]:
fig=plt.figure(figsize=(12, 12))

for i in range(2,6):
    fig.add_subplot(2, 2, i-1)
    plt.imshow(labels2rgb(labels_train[i]), cmap=color_map)

plt.show()

Let us check the distribution of the segmentation classes in both AOIs 

In [ ]:
cls_train = np.argmax(labels_train, axis=-1)

cls_test = np.argmax(labels_test, axis=-1)

plt.hist([cls_train.flatten(), cls_test.flatten()], label=["Train", "Test"], density=True, bins=np.arange(0,8)-0.5)
plt.legend()
plt.show()

Mmmm... they look quite different... 

## 2. Classification task

We will build a classification data set using the planet data. First, we will get 100x100 snaps from the images.

In [ ]:
patches_X = np.lib.stride_tricks.sliding_window_view(planet_train[:,:1000,:1000,:],
                                        [1,50,50,4])[::1, ::50, ::50, ::4]
patches_X = patches_X.reshape(-1, 50, 50, 4)

print("Patches shape:", patches_X.shape)
plt.imshow(true_color(patches_X[2]))

Then, we will repeat the process to get the corresponding 100x100 segmentation mask.

In [ ]:
patches_Y = np.lib.stride_tricks.sliding_window_view(labels_train[:,:1000,:1000,:],
                                        [1,50,50,7])[::1, ::50, ::50, ::7]

patches_Y = patches_Y.reshape(-1, 50, 50, 7)

print("Labels shape:", patches_Y.shape)
plt.imshow(labels2rgb(patches_Y[2]), cmap=color_map)

Some models expect "categorical" classes, instead of one-hot encoded.

In [ ]:
patches_Y_cat = np.argmax(patches_Y, axis=-1)

Now, we will create an image classification data set. In particular, we will classify each image based on the majority of the pixels.

In [ ]:
patches_Y_majority = list()

for y in patches_Y_cat:
    cat, counts = np.unique(y, return_counts=True)
    patches_Y_majority.append( int(cat[np.argmax(counts)]) )

patches_Y_majority = np.array(patches_Y_majority)

We split the data into train and test.

In [ ]:
train_indices, test_indices = train_test_split(np.arange(patches_X.shape[0]), test_size=0.2, random_state=42)

X_train = patches_X[train_indices,:,:]
X_test = patches_X[test_indices,:,:]

Y_train = patches_Y_majority[train_indices]
Y_test = patches_Y_majority[test_indices]

In [ ]:
plt.hist([Y_train, Y_test], label=["Train", "Test"], density=True, bins=np.arange(0,9)-0.5)
plt.legend()
plt.show()

### 2.1. Random Forest Classifier

Machine learning: A computer program is said to learn from experience E with respect to some task T and some performance measure P, if its performance on T, as measured by P, improves with experience E (*Tom M. Michell, 1997*).


A random forest is a meta estimator that fits a number of classifying decision trees on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.

![Random Forest](random-forest.png)

In [ ]:
# We need to convert the images to vectors
X_train_flat = X_train.reshape(X_train.shape[0], math.prod(X_train.shape[1:]))
X_test_flat = X_test.reshape(X_test.shape[0], math.prod(X_test.shape[1:]))

In [ ]:
rf_classifier = RandomForestClassifier(random_state=0)

rf_classifier.fit(X_train_flat, Y_train)

In [ ]:
Y_pred_rf = rf_classifier.predict(X_test_flat)

ConfusionMatrixDisplay.from_predictions(Y_test, Y_pred_rf)
plt.show()

acc_rf = accuracy_score(Y_test, Y_pred_rf)
print("Accuracy Random Forest Classifier: ", acc_rf)

### 2.2. Support Vector Machine

There are multiple ML models for classification, including both supervised and unsupervised methods. For example, here you can check several classifiers being compared.



In [ ]:
svm_classifier = SVC()
svm_classifier.fit(X_train_flat, Y_train)

#### Do it yourself

Predict using the SVM and analyse the results. How does the performance compare to the RF?

In [ ]:
# predict and analyse the results


### 2.3. Artificial Neural Network

An artificial neural network (ANN) is a network of computer units (neurons) connected to each other, where each unit performs a calculation from an input and communicates its result (output) to the connected units. Typically, the connections are weighted, which indicates how strong the connection between two units is. Also, the calculation units are usually grouped into modules or layers. ANNs have properties and capabilities that are very useful for solving complex problems, highlighting its nonlinearity, adaptability, fault tolerance, among others. Thanks to these properties, it has been possible to overcome the results obtained in multiple problems.

![Artificial neural network](ann.png)
    
Deep-learning methods are representation-learning methods with multiple levels of representation, obtained by composing simple but non-linear modules that each transform the representation at one level (starting with the raw input) into a representation at a higher, slightly more abstract level. With the composition of enough such transformations, very complex functions can be learned. *Yann LeCun, Yoshua Bengio and Geoffrey Hinton, 2015*

![Deep Learning](deep-learning.png)


In [ ]:
class NumpyImageDataset(Dataset):
    def __init__(self, images_np, labels_np):
        self.images = images_np
        self.labels = labels_np

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Fetch a single sample from the numpy array
        img = self.images[idx]     
        label = self.labels[idx]
        
        # Step A: Convert HWC (NumPy) to CHW (PyTorch) layout
        # (Axes: 0=Height, 1=Width, 2=Channels -> New positions: 2, 0, 1)
        img = np.transpose(img, (2, 0, 1)) # Shape: [channels, height, width]
        
        # Step B: Convert to PyTorch Tensors
        # torch.from_numpy shares memory, making it incredibly fast
        img_tensor = torch.from_numpy(img).float()
        label_tensor = torch.tensor(label, dtype=torch.long)
        
        return img_tensor, label_tensor


dataset = NumpyImageDataset(X_train, Y_train)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

test_dataset = NumpyImageDataset(X_test, Y_test)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
class SimpleImageCNN(nn.Module):
    def __init__(self, in_channels=4, num_classes=7):
        super(SimpleImageCNN, self).__init__()
        
        # 1. Convolutional Layers (Extract Spatial Features)
        # Conv1: input channels, 16 output filters, 3x3 kernel
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
        # Conv2: 16 input filters, 32 output filters, 3x3 kernel
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        # 2. Max Pooling (Reduces image dimensions by half)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 3. Fully Connected Layers (Classification Head)
        # Image patch size of 50x50 pixels:
        # After two MaxPool layers, 50x50 becomes 12x12. 
        # Feature size = 32 filters * 12 * 12 pixels = 8192
        self.fc1 = nn.Linear(32 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Input 'x' shape: [Batch_Size, Channels, Height, Width]
        
        # First Block: Conv -> ReLU -> Pool
        x = self.pool(torch.relu(self.conv1(x))) # Size drops from 50x50 to 25x25
        
        # Second Block: Conv -> ReLU -> Pool
        x = self.pool(torch.relu(self.conv2(x))) # Size drops from 25x25 to 12x12
        
        # Flatten the 3D features into a 1D vector for the linear layers
        x = x.view(x.size(0), -1) 
        
        # Classification layers
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleImageCNN()

summary(model, input_size=(1, 4, 50, 50))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loss_history = []

model.train()
for epoch in range(5):
    running_loss = 0.0
    for images, seg_labels in dataloader:
        # Move data tensors onto the target compute hardware (CPU or GPU)
        images, seg_labels = images.to(device), seg_labels.to(device)
        
        # Reset optimization gradients
        optimizer.zero_grad()
        
        # Forward pass & loss evaluation
        outputs = model(images)
        loss = criterion(outputs, seg_labels)
        
        # Backpropagation & weight updating
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(dataloader.dataset)
    print(f"Epoch [{epoch+1}/5] Completed - Average Loss: {epoch_loss:.4f}")
    train_loss_history.append(epoch_loss)


In [ ]:
model.eval()

test_loss = 0.0
correct_predictions = 0
total_samples = 0
Y_pred_simple_cnn = []

# Deactivate the gradient tracking engine to save memory
with torch.no_grad():
    for images, targets in test_dataloader:
        # Move data tensors onto the target compute hardware (CPU or GPU)
        images, targets = images.to(device), targets.to(device)
        
        # Forward pass: Generate predictions
        outputs = model(images)
        
        # Calculate loss on the batch
        loss = criterion(outputs, targets)
        test_loss += loss.item() * images.size(0)
        
        # Extract the index of the highest logit score (the predicted class)
        # outputs shape: [Batch_Size, Num_Classes]
        _, predicted_classes = torch.max(outputs, dim=1)
        Y_pred_simple_cnn.extend(predicted_classes.cpu().numpy())
        
        # Update metrics
        total_samples += targets.size(0)
        correct_predictions += (predicted_classes == targets).sum().item()

test_accuracy = (correct_predictions / total_samples) * 100

ConfusionMatrixDisplay.from_predictions(Y_test, Y_pred_simple_cnn)
plt.show()

print("Accuracy Simple CNN: ", test_accuracy)

In [ ]:
epochs = range(1, len(train_loss_history) + 1)

plt.figure()
plt.plot(epochs, train_loss_history, 'b', label='Training loss')
plt.show()

#### Do it yourself

What happen if you repeat the experiment? Are you getting the same results? Why is this happening?

### 2.4. Data Preprocessing

Deep learning models are sensitive to the hyperparameters and data dependent. Therefore, it is a good idea to spend some time preparing the data and tailoring the model.

In [ ]:
planet_scaler = StandardScaler()
# the scaler expects data to have only two dimensions (i.e., a set of vectors)
planet_scaler.fit(X_train_flat)

# we want to use the data without loosing spatial information
X_train_flat_std = planet_scaler.transform(X_train_flat)
X_train_std = X_train_flat_std.reshape(X_train.shape)

# repeat for the test data
X_test_flat_std = planet_scaler.transform(X_test_flat)
X_test_std = X_test_flat_std.reshape(X_test.shape)



train_dataset_std = NumpyImageDataset(X_train_std, Y_train)
train_dataloader_std = DataLoader(train_dataset_std, batch_size=8, shuffle=True)

test_dataset_std = NumpyImageDataset(X_test_std, Y_test)
test_dataloader_std = DataLoader(test_dataset_std, batch_size=8, shuffle=False)

In [ ]:
plt.figure()
plt.title("Original")
plt.hist(planet_train[:,:,:,0].flatten(), color="blue", label="blue", alpha=0.5, bins=100)
plt.hist(planet_train[:,:,:,1].flatten(), color="green", label="green", alpha=0.5, bins=100)
plt.hist(planet_train[:,:,:,2].flatten(), color="red", label="red", alpha=0.5, bins=100)
plt.hist(planet_train[:,:,:,3].flatten(), color="orange", label="nir", alpha=0.2, bins=100)
plt.legend()


plt.figure()
plt.title("Standardized")
plt.hist(X_train_std[:,:,:,0].flatten(), color="blue", label="blue", alpha=0.5, bins=100)
plt.hist(X_train_std[:,:,:,1].flatten(), color="green", label="green", alpha=0.5, bins=100)
plt.hist(X_train_std[:,:,:,2].flatten(), color="red", label="red", alpha=0.5, bins=100)
plt.hist(X_train_std[:,:,:,3].flatten(), color="orange", label="nir", alpha=0.2, bins=100)
# plt.yscale('symlog')
plt.legend()
plt.show()

In [ ]:
model_2 = SimpleImageCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_2.parameters(), lr=0.001)

train_loss_history_2 = []

model_2.train()
for epoch in range(5):
    running_loss = 0.0
    for images, seg_labels in train_dataloader_std:
        # Move data tensors onto the target compute hardware (CPU or GPU)
        images, seg_labels = images.to(device), seg_labels.to(device)
        
        # Reset optimization gradients
        optimizer.zero_grad()
        
        # Forward pass & loss evaluation
        outputs = model_2(images)
        loss = criterion(outputs, seg_labels)
        
        # Backpropagation & weight updating
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(dataloader.dataset)
    print(f"Epoch [{epoch+1}/5] Completed - Average Loss: {epoch_loss:.4f}")
    train_loss_history_2.append(epoch_loss)

In [ ]:
model_2.eval()

test_loss = 0.0
correct_predictions = 0
total_samples = 0
Y_pred_simple_cnn_2 = []

# Deactivate the gradient tracking engine to save memory
with torch.no_grad():
    for images, targets in test_dataloader_std:
        # Move data tensors onto the target compute hardware (CPU or GPU)
        images, targets = images.to(device), targets.to(device)
        
        # Forward pass: Generate predictions
        outputs = model_2(images)
        
        # Calculate loss on the batch
        loss = criterion(outputs, targets)
        test_loss += loss.item() * images.size(0)
        
        # Extract the index of the highest logit score (the predicted class)
        # outputs shape: [Batch_Size, Num_Classes]
        _, predicted_classes = torch.max(outputs, dim=1)
        Y_pred_simple_cnn_2.extend(predicted_classes.cpu().numpy())
        
        # Update metrics
        total_samples += targets.size(0)
        correct_predictions += (predicted_classes == targets).sum().item()

test_accuracy_2 = (correct_predictions / total_samples) * 100

ConfusionMatrixDisplay.from_predictions(Y_test, Y_pred_simple_cnn_2)
plt.show()

print("Accuracy Simple CNN: ", test_accuracy_2)

#### Do it yourself

For most neural network architectures and problems, a Standard Scaler is the preferred default choice. However, in certain scenarios a MinMax or Uniform Scaler could yield better results. How about in this case? 

In [ ]:
planet_minmax_scaler = MinMaxScaler()

# TODO continue implementing the experiment

### 2.5. Spectral Information


In [ ]:
patches_X_s2 = np.lib.stride_tricks.sliding_window_view(sen2_train[:,:1000,:1000,:],
                                        [1,50,50,12])[::1, ::50, ::50, ::4]
patches_X_s2 = patches_X_s2.reshape(-1, 50, 50, 12)

print("Patches shape:", patches_X_s2.shape)
plt.imshow(true_color(patches_X_s2[2]))

X_train_s2 = patches_X_s2[train_indices,:,:]
X_test_s2 = patches_X_s2[test_indices,:,:]

dataset_s2 = NumpyImageDataset(X_train_s2, Y_train)
dataloader_s2 = DataLoader(dataset_s2, batch_size=8, shuffle=True)

test_dataset_s2 = NumpyImageDataset(X_test_s2, Y_test)
test_dataloader_s2 = DataLoader(test_dataset_s2, batch_size=8, shuffle=False)

In [ ]:
model_s2 = SimpleImageCNN(in_channels=12)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_s2.parameters(), lr=0.001)

train_loss_history = []

model_s2.train()
for epoch in range(5):
    running_loss = 0.0
    for images, seg_labels in dataloader_s2:
        # Move data tensors onto the target compute hardware (CPU or GPU)
        images, seg_labels = images.to(device), seg_labels.to(device)
        
        # Reset optimization gradients
        optimizer.zero_grad()
        
        # Forward pass & loss evaluation
        outputs = model_s2(images)
        loss = criterion(outputs, seg_labels)
        
        # Backpropagation & weight updating
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(dataloader.dataset)
    print(f"Epoch [{epoch+1}/5] Completed - Average Loss: {epoch_loss:.4f}")
    train_loss_history.append(epoch_loss)


In [ ]:
model_s2.eval()

test_loss = 0.0
correct_predictions = 0
total_samples = 0
Y_pred_simple_cnn = []

# Deactivate the gradient tracking engine to save memory
with torch.no_grad():
    for images, targets in test_dataloader_s2:
        # Move data tensors onto the target compute hardware (CPU or GPU)
        images, targets = images.to(device), targets.to(device)
        
        # Forward pass: Generate predictions
        outputs = model_s2(images)
        
        # Calculate loss on the batch
        loss = criterion(outputs, targets)
        test_loss += loss.item() * images.size(0)
        
        # Extract the index of the highest logit score (the predicted class)
        # outputs shape: [Batch_Size, Num_Classes]
        _, predicted_classes = torch.max(outputs, dim=1)
        Y_pred_simple_cnn.extend(predicted_classes.cpu().numpy())
        
        # Update metrics
        total_samples += targets.size(0)
        correct_predictions += (predicted_classes == targets).sum().item()

test_accuracy_s2 = (correct_predictions / total_samples) * 100

ConfusionMatrixDisplay.from_predictions(Y_test, Y_pred_simple_cnn)
plt.show()


global_baseline_loss = test_loss / total_samples
global_baseline_acc = correct_predictions / total_samples

print("Accuracy Simple CNN: ", test_accuracy_s2)

Are all bands equal? Let us check the band importance using a technique called **Permutation Feature Importance**. This is a model-agnostic technique used to measure how much a trained machine learning model relies on a specific input feature (or spectral band) to make accurate predictions.

In simple terms, we will re-evaluate the performance of the model *shuffeling* each of the bands accross the dataset, i.e., we will select a band (or feature), and then we will swap the whole band information among the different images. All the data will still be there, we are not altering the data distribution, but the selected band became *noise*, as it is not assigned to the correct label.

If the band is highly important, there will be a drastical drop in the performance.

In [ ]:
band_names = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]

band_loss_increases = []
band_accuracy_drops = []

num_channels = len(band_names)

for channel_idx in range(num_channels):
    total_permuted_loss = 0.0
    total_permuted_correct = 0
    
    with torch.no_grad():
        for images, labels in test_dataloader_s2:
            images, labels = images.to(device), labels.to(device)
            
            # Clone the current batch to avoid altering data in the loader
            images_permuted = images.clone()
            
            # Shuffle the spatial layer of channel_idx across the batch dimension
            shuffle_idx = torch.randperm(images_permuted.size(0))
            images_permuted[:, channel_idx, :, :] = images_permuted[shuffle_idx, channel_idx, :, :]
            
            # Evaluate model on permuted batch
            outputs_permuted = model_s2(images_permuted)
            loss_permuted = criterion(outputs_permuted, labels)
            total_permuted_loss += loss_permuted.item() * images.size(0)
            
            preds_permuted = torch.argmax(outputs_permuted, dim=1)
            total_permuted_correct += (preds_permuted == labels).sum().item()
    
    # Aggregate performance metrics for this specific band
    global_permuted_loss = total_permuted_loss / total_samples
    global_permuted_acc = total_permuted_correct / total_samples

    # Quantify the drop in capability (forcing negative values to 0)
    loss_increase = max(0.0, global_permuted_loss - global_baseline_loss)
    acc_drop = max(0.0, global_baseline_acc - global_permuted_acc)
    
    band_loss_increases.append(loss_increase)
    band_accuracy_drops.append(acc_drop)    

In [ ]:
importance_df = pd.DataFrame({
    'Spectral Band': band_names,
    'Delta Loss Increase': band_loss_increases,
    'Delta Accuracy Drop': band_accuracy_drops
}).sort_values(by='Delta Loss Increase', ascending=False)

print(f"Baseline Loss: {global_baseline_loss:.4f} | Baseline Accuracy: {global_baseline_acc * 100:.2f}%")
print("\nBand Importance Rankings")
print(importance_df.to_string(index=False))

### Do it yourself

In remote sensing bands are often highly correlated (e.g., the Red band and the Red Edge band often move together). Thus, this technique may fail. Alternatives? Many! One simple and similar approach is **Feature Dropping**. In this case, instead of shuffeling the bands, one band is zeroed.